In [1]:
import os
import sys

# Colab/repo setup. This notebook imports the clean baseline helper module
# `legacy_retrieval_engine.py`, so the runtime must be inside the repo or have
# the repo on sys.path.
repo_url = "https://github.com/allarom/advanced-genai-26.git"
repo_dir = "advanced-genai-26"
branch_name = "main"  # change only if testing a different remote branch

current_dir = os.path.abspath(os.getcwd())
if os.path.basename(current_dir) == repo_dir and os.path.exists(os.path.join(current_dir, ".git")):
    repo_path = current_dir
else:
    repo_path = os.path.abspath(repo_dir)
    if not os.path.exists(repo_path):
        !git clone -b {branch_name} {repo_url} {repo_path}

# Ensure an already-cloned Colab repo is on the intended branch and up to date.
%cd {repo_path}
!git fetch origin
!git checkout {branch_name}
!git pull origin {branch_name}

repo_path = os.path.abspath(os.getcwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

helper_path = os.path.join(repo_path, "legacy_retrieval_engine.py")
if not os.path.exists(helper_path):
    raise FileNotFoundError(
        f"legacy_retrieval_engine.py was not found at {helper_path}. "
        f"Make sure it is committed and pushed to the remote branch '{branch_name}'."
    )

print("✅ Project environment synchronized.")
print("CWD:", os.getcwd())
print("Branch:")
!git branch --show-current
print("Helper exists:", os.path.exists(helper_path))


Cloning into '/content/advanced-genai-26'...
remote: Enumerating objects: 587, done.
remote: Counting objects: 100% (242/242), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 587 (delta 105), reused 111 (delta 54), pack-reused 345 (from 1)
Receiving objects: 100% (587/587), 10.16 MiB | 21.54 MiB/s, done.
Resolving deltas: 100% (320/320), done.
/content/advanced-genai-26
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/allarom/advanced-genai-26
 * branch            main       -> FETCH_HEAD
Already up to date.
✅ Project environment synchronized.
CWD: /content/advanced-genai-26
Branch:
main
Helper exists: True


# Reliable and Adaptive Agentic RAG System (Step 3)

This notebook extends the Step 2 multi-agent retrieval system with reliability, adaptation, recovery, and trust mechanisms.

The goal is to make the RAG system more reliable when evidence is:

weak
incomplete
ambiguous
contradictory

The notebook implements multiple reliability mechanisms and adaptive orchestration behaviors.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
print(f'Current working directory: {os.getcwd()}')
!ls -a

Current working directory: /content/advanced-genai-26
.			    README.md
..			    report.md
archived_documents	    reports
baseline		    scripts
baseline_repro_report.md    Step_1_Baseline_and_Failure_Analysis.ipynb
.git			    Step_2_Reliability_Aware_Design.ipynb
.gitignore		    Step_3_Reliable_Adaptive_Agentic_RAG.ipynb
legacy_retrieval_engine.py  Step_4_1_extra_challenges.ipynb
memory			    Step_4_Evaluation.ipynb


In [4]:
# Verify that the clean legacy retrieval module is available from the repo root
import os
import sys

repo_path = os.path.abspath(os.getcwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

helper_path = os.path.join(repo_path, "legacy_retrieval_engine.py")
if os.path.exists(helper_path):
    print('✅ legacy_retrieval_engine.py found. Step 3 can import the retrieval engine.')
    print('CWD:', os.getcwd())
else:
    print('❌ legacy_retrieval_engine.py NOT found in the current repo directory.')
    print('CWD:', os.getcwd())


✅ legacy_retrieval_engine.py found. Step 3 can import the retrieval engine.
CWD: /content/advanced-genai-26


## 1. Installation

In [5]:
# Install dependencies required by the baseline retrieval helper and Step 3.
# `legacy_retrieval_engine.py` imports langdetect, sentence-transformers,
# LangChain vector-store integrations, Chroma, NLTK, and pytrec_eval.
!apt-get update -qq
!apt-get install -y -qq libtrec-dev
!pip install -q     pandas numpy nltk langdetect rank-bm25     transformers accelerate sentence-transformers     langchain-core langchain-community langchain-huggingface chromadb     pytrec_eval


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package libtrec-dev
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 59.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 

## 2. Imports

In [6]:
import re
import time
import random
import numpy as np
import pandas as pd

from collections import defaultdict

## 3. Load Baseline Retrieval Helpers

This section imports the orchestration strategies and retrievers from `legacy_retrieval_engine.py`, the clean helper module for the reproduced baseline retrieval system.


In [7]:
import importlib.util
import subprocess
import sys

# Dependency guard for Colab: install helper dependencies if the install cell was skipped
# or if the runtime was restarted after installing packages.
_required_packages = {
    "langdetect": "langdetect",
    "sentence_transformers": "sentence-transformers",
    "langchain_core": "langchain-core",
    "langchain_community": "langchain-community",
    "langchain_huggingface": "langchain-huggingface",
    "chromadb": "chromadb",
    "nltk": "nltk",
    "rank_bm25": "rank-bm25",
}
_missing_packages = [pip_name for import_name, pip_name in _required_packages.items()
                     if importlib.util.find_spec(import_name) is None]
if _missing_packages:
    print("Installing missing packages:", _missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing_packages])

from legacy_retrieval_engine import (
    WEIGHT_PRESETS,
    K_VALUES,
    RETRIEVE_K,
    TOP_K,
    orchestrator,
    waterfall_orchestrator,
    voting_orchestrator,
    confidence_orchestrate,
    waterfall_orchestrate,
    voting_orchestrate,
)

print('Legacy retrieval engine loaded from legacy_retrieval_engine.py.')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Legacy retrieval engine loaded from legacy_retrieval_engine.py.


In [8]:
# Inspect the lengths of retrieved chunks (documents).
# This cell is self-contained: if `docs` does not exist yet, it runs a small
# retrieval call first.
if "docs" not in globals():
    sample_query = "who at eth received erc grants?"
    answer, docs, trace = confidence_orchestrate(sample_query, top_k=5)
    print(f"Created docs using sample query: {sample_query}")

print(f"Number of documents retrieved: {len(docs)}")
for i, d in enumerate(docs[:5]):
    content = d.page_content
    char_count = len(content)
    word_count = len(content.split())
    print(f"Doc {i} length: {char_count} characters (~{word_count} words)")
    print(f"Snippet: {content[:100]}...")
    print("-" * 30)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Created docs using sample query: who at eth received erc grants?
Number of documents retrieved: 5
Doc 0 length: 2616 characters (~408 words)
Snippet: passage: strong performance high expectations: large sum of research funds staying in the top league...
------------------------------
Doc 1 length: 1431 characters (~248 words)
Snippet: erc advanc grant for two eth research vorholt professor of microbiolog has been award a second erc a...
------------------------------
Doc 2 length: 2298 characters (~389 words)
Snippet: erc synergi grant eth and europ conduct cuttingedg research togeth this year eth zurich has alreadi ...
------------------------------
Doc 3 length: 2314 characters (~421 words)
Snippet: erc advanc grant for two eth research european research is more import than ever detlef günther eth ...
------------------------------
Doc 4 length: 2358 characters (~416 words)
Snippet: erc advanc grant eth must remain attract the win project came from research field in which eth zuric..

## 4. Utility Functions

In [9]:
def normalize(text):
    """Cleans text and converts it into a set of unique lowercase words."""
    # Convert to lowercase to ensure case-insensitive matching
    text = text.lower()
    # Remove punctuation using regex, replacing it with spaces
    text = re.sub(r"[^\w\s]", " ", text)
    # Split into words and return a set for fast overlap calculation
    return set(text.split())

def overlap_score(a, b):
    """Calculates the percentage of words in 'a' that are also found in 'b'."""
    # Normalize both strings into sets of tokens
    ta = normalize(a)
    tb = normalize(b)

    # Prevent division by zero if string 'a' is empty
    if len(ta) == 0:
        return 0.0

    # Intersection (ta & tb) finds common words; divide by total in 'a'
    return len(ta & tb) / len(ta)

## 5. Mechanisms

### 5.1 Evidence Sufficiency Estimation (A)
This module estimates whether the retrieved evidence is sufficient.

Signals used:

- overlap between query and retrieved chunks
- number of supporting chunks
- retrieval fusion score

In [10]:
class EvidenceSufficiencyAgent:
    """
    Checks: Do we have enough evidence to answer this question?

    How it works:
    - Count how many query words appear in each retrieved document.
    - Calculate average overlap across top 5 docs.
    - If average >= 0.15, evidence is considered sufficient.

    Why it matters:
    Prevents the system from answering when retrieval returns
    irrelevant or off-topic documents.
    """

    def assess(self, query, docs):

        if len(docs) == 0:
            return {
                "sufficient": False,
                "score": 0.0,
            }

        q_tokens = normalize(query)

        support_scores = []

        for d in docs[:5]:

            txt = d.page_content
            d_tokens = normalize(txt)

            overlap = len(q_tokens & d_tokens)
            overlap = overlap / max(len(q_tokens), 1)

            support_scores.append(overlap)

        avg_support = np.mean(support_scores)

        sufficient = avg_support >= 0.15

        return {
            "sufficient": sufficient,
            "score": round(float(avg_support), 3),
            "supporting_chunks": len([
                x for x in support_scores if x > 0.1
            ])
        }


### %.2 Groundness / Support Verification (B)
The answer is verified against retrieved evidence.

In [11]:
class GroundednessAgent:
    """
    Checks: Is the answer actually supported by the retrieved documents?

    How it works (dual-threshold):
    1. Global check: answer words must overlap >= 45% with ALL docs combined
    (more than 45% words appear in all docs).
       (ensures the answer is broadly supported across evidence)
    2. Per-doc check: at least ONE doc must share >= 25% of answer words.
       (ensures the answer is strongly supported by at least one source)
       'Share' means that a single document must contain enough of the answer's key vocabulary
    Both thresholds must pass. Stopwords (the, is, of...) are ignored.

    Why it matters:
    Prevents hallucination — answers that sound plausible but are not
    actually found in the source documents.
    """

    # Lightweight stopword list for cleaner token overlap
    _STOP = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
        'shall', 'should', 'can', 'could', 'may', 'might', 'must',
        'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from', 'as', 'to',
        'and', 'or', 'but', 'if', 'then', 'than', 'so', 'yet',
        'it', 'its', 'this', 'that', 'these', 'those',
        'i', 'you', 'he', 'she', 'we', 'they', 'me', 'him', 'her',
        'us', 'them', 'my', 'your', 'his', 'our', 'their',
        'what', 'which', 'who', 'when', 'where', 'why', 'how',
    }

    def _tokens(self, text):
        """Return set of non-stopword tokens."""
        return normalize(text) - self._STOP

    def verify(self, answer, docs,
               min_global_overlap=0.45,
               min_per_doc_overlap=0.25):

        ans_tokens = self._tokens(answer)

        if len(ans_tokens) == 0:
            return False

        # --- global overlap: answer vs union of all docs ---
        all_doc_tokens = set()
        per_doc_overlaps = []

        for d in docs[:5]:
            txt = d.page_content or ''
            doc_tokens = self._tokens(txt)
            all_doc_tokens |= doc_tokens
            per_doc = len(ans_tokens & doc_tokens) / max(len(ans_tokens), 1)
            per_doc_overlaps.append(per_doc)

        global_overlap = len(ans_tokens & all_doc_tokens) / max(len(ans_tokens), 1)
        best_doc_overlap = max(per_doc_overlaps) if per_doc_overlaps else 0.0

        grounded = (
            global_overlap >= min_global_overlap and
            best_doc_overlap >= min_per_doc_overlap
        )

        return grounded



### 5.3 Contradiction Detection (C)
This mechanism checks whether retrieved chunks contain conflicting statements.

Lightweight heuristic:

- detect conflicting keywords
- detect opposite numeric claims

In [12]:
import re

class ContradictionAgent:
    """
    Checks: Do the retrieved documents contradict each other?

    How it works (two heuristics):
    1. Antonym keywords (word-boundary matched, so 'no' does NOT match
       'Nobel' or 'north'). Pairs like increase/decrease, true/false.
    2. Year conflict: if focused documents (each mentioning exactly one
       year) cite DIFFERENT years, the sources disagree on 'when'.

    Why it matters:
    Conflicting evidence produces unreliable answers. We lower trust so
    the system can recover or abstain instead of guessing.
    Future upgrade: LLM-based semantic contradiction detection.
    """

    CONTRADICTIONS = [
        ("yes", "no"),
        ("increase", "decrease"),
        ("increased", "decreased"),
        ("rose", "fell"),
        ("higher", "lower"),
        ("true", "false"),
        ("approved", "rejected"),
    ]

    def _has_word(self, word, text):
        # \\b = word boundary, so we match whole words only.
        return re.search(r"\b" + re.escape(word) + r"\b", text) is not None

    def _single_year(self, text):
        years = re.findall(r"\b(?:1\d{3}|20\d{2})\b", text)
        return years[0] if len(set(years)) == 1 and years else None

    def detect(self, docs):

        texts = [d.page_content.lower() for d in docs[:5]]

        # Heuristic 1: antonym keyword conflict (whole-word match)
        for a, b in self.CONTRADICTIONS:

            has_a = any(self._has_word(a, t) for t in texts)
            has_b = any(self._has_word(b, t) for t in texts)

            if has_a and has_b:
                return {
                    "contradiction": True,
                    "reason": f"Detected conflict: {a} vs {b}"
                }

        # Heuristic 2: focused docs disagree on the year
        single_years = [y for y in (self._single_year(t) for t in texts) if y]

        if len(set(single_years)) > 1:
            return {
                "contradiction": True,
                "reason": f"Date conflict across sources: {sorted(set(single_years))}"
            }

        return {
            "contradiction": False,
            "reason": "No obvious contradiction detected"
        }

### 5.4 Clarification Strategy (D)
The system detects ambiguous or underspecified questions.

In [13]:
class ClarificationAgent:
    """
    Checks: Is the user's question clear enough to answer?

    How it works:
    - Too short? (<= 3 words) → likely underspecified.
    - Contains vague pronouns? ('it', 'they', 'this') → ambiguous reference.
    If either rule matches, ask the user to clarify.

    Why it matters:
    Prevents the system from guessing when the user's intent is unclear.
    Better to ask than to answer the wrong question.
    """

    def needs_clarification(self, query):

        q = query.lower().strip()

        ambiguous = [
            "it",
            "they",
            "this",
            "that",
        ]

        short_query = len(q.split()) <= 3

        ambiguous_ref = any(x in q.split() for x in ambiguous)

        if short_query or ambiguous_ref:
            return True

        return False

    def clarification_question(self, query):

        return (
            "Could you clarify your question or provide "
            "more specific details?"
        )


### 5.5 Abstention Mechanism (E)
The system abstains when evidence is weak or contradictory.

In [14]:
class AbstentionAgent:
    """
    Decides: Should the system refuse to answer?

    How it works (Step 2 §2.3):
    Abstains when the overall TRUST score falls below a threshold (0.4).
    The trust score already combines evidence sufficiency, groundedness,
    and contradiction, so a single threshold captures every failure mode:
    - low sufficiency  -> low trust
    - not grounded     -> low trust
    - contradiction    -> low trust

    Why it matters:
    A system that knows when it does NOT know is more trustworthy than one
    that always guesses. This is the 'I don't know' safety net, and using
    the trust score keeps the decision consistent with the trace signals.
    """

    def should_abstain(self, trust, threshold=0.4):

        return trust["score"] < threshold

### 5.6 Self-Reflection / Critique Loop (F)
The critic reviews the draft answer.

In [15]:
import re

class CriticAgent:
    """
    Reviews the draft answer and reports any quality issues.

    Checks performed:
    1. Grounding: flags if answer is weakly supported by documents.
    2. Contradiction: flags if evidence conflicts.
    3. Length: flags if answer is suspiciously short (< 3 words).
    4. Temporal coherence (entity_temporal queries only):
       - Finds all years mentioned in the answer.
       - Checks if at least one year is within +/- 10 of the query year.
       - Catches answers that reference the wrong decade entirely.

    Why it matters:
    The Critic is the final quality gate before the answer reaches the user.
    It catches subtle errors that other agents might miss.
    """

    _YEAR_RE = re.compile(r'\b(1\d{3}|20\d{2})\b')

    def _temporal_coherent(self, answer, query_year, window=10):
        """
        Answer must mention at least one year within +/- window of query_year.
        Catches answers that are grounded but reference the wrong decade.
        """
        found = [int(m) for m in self._YEAR_RE.findall(answer)]
        return any(abs(y - query_year) <= window for y in found)

    def critique(self,
                 answer,
                 grounded,
                 contradiction,
                 query_type=None,
                 query_year=None):

        feedback = []

        if not grounded:
            feedback.append("Answer weakly supported")

        if contradiction["contradiction"]:
            feedback.append("Evidence conflict detected")

        if len(answer.split()) < 3:
            feedback.append("Answer too short")

        # Temporal coherence check for entity_temporal queries
        if query_type == 'entity_temporal' and query_year:
            temporal_ok = self._temporal_coherent(answer, query_year, window=10)
            if not temporal_ok:
                feedback.append(
                    f"Temporal mismatch: answer references wrong time period (expected around {query_year})"
                )

        if len(feedback) == 0:
            feedback.append("Answer appears acceptable")

        return feedback

### 5.7 Recovery Mechanism (G)
The system changes behavior dynamically.

Recovery actions:

- switch retrieval strategy
- rewrite query
- move to clarification mode
- abstain

In [16]:
class RecoveryAgent:
    """
    Decides HOW to fix a low-reliability result, and the orchestrator
    executes that action (re-retrieve). This is the 'adaptive' part.

    Action priority (checks the WHY behind low trust):
    1. contradiction  -> switch_strategy to 'voting' (cross-checks sources)
    2. not grounded   -> switch_strategy to 'waterfall' (escalates retrievers
                         BM25 -> +Dense -> +Graph to broaden evidence)
    3. not sufficient -> rewrite_query (add context to improve retrieval)
    4. otherwise      -> none

    Note: groundedness is now considered (it was ignored before). Switching
    to 'waterfall' is the Step-3-layer equivalent of the old CriticAgent's
    'broaden retrieval' retry, since retriever weights live inside the
    orchestrator and cannot be tuned directly from here.

    Why it matters:
    Instead of giving up, the system tries to fix the problem once before
    abstaining. This turns 'reliable' into 'reliable AND adaptive'.
    """

    def rewrite_query(self, query):

        return query + " ETH Zurich"

    def recover(self,
                query,
                current_strategy,
                sufficiency,
                contradiction,
                grounded=True):

        if contradiction["contradiction"]:

            return {
                "action": "switch_strategy",
                "new_strategy": "voting"
            }

        if not grounded:

            fallback = "waterfall" if current_strategy != "waterfall" else "voting"

            return {
                "action": "switch_strategy",
                "new_strategy": fallback
            }

        if not sufficiency["sufficient"]:

            rewritten = self.rewrite_query(query)

            return {
                "action": "rewrite_query",
                "query": rewritten
            }

        return {
            "action": "none"
        }

### 5.8 Trust / Confidence Scoring (H)
The confidence score combines:

- evidence sufficiency
- groundedness
- contradiction detection

In [17]:
class TrustAgent:
    """
    Computes a single confidence score from all reliability signals.

    Formula:
    trust = 0.6 * sufficiency_score + 0.3 * groundedness_bonus - 0.4 * contradiction_penalty

    Score ranges:
    - HIGH (> 0.7): strong evidence, confident answer.
    - MEDIUM (0.4-0.7): acceptable but not perfect.
    - LOW (< 0.4): weak evidence, consider abstaining.

    Why it matters:
    Turns multiple checks into one easy-to-understand number.
    Users and downstream systems can act on this single score.
    """

    def score(self,
              sufficiency,
              grounded,
              contradiction):

        score = 0.0

        score += sufficiency["score"] * 0.6

        if grounded:
            score += 0.3

        if contradiction["contradiction"]:
            score -= 0.4

        score = max(0.0, min(1.0, score))

        if score > 0.7:
            label = "HIGH"
        elif score > 0.4:
            label = "MEDIUM"
        else:
            label = "LOW"

        return {
            "score": round(float(score), 3),
            "label": label
        }


## 6. Adaptive Reliable RAG Orchestrator
This orchestrator integrates all reliability mechanisms.

In [18]:
class ReliableAdaptiveRAG:
    """
    Main orchestrator: wraps the Step 2 retrieval engine with
    8 reliability agents to produce trustworthy answers.

    Decision flow (4 branches):
    1. CLARIFY → if query is ambiguous (short / pronouns)
    2. ABSTAIN → if evidence is weak, ungrounded, or contradictory
    3. RECOVER → if first attempt fails, retry with new strategy/query
    4. ANSWER → if all checks pass, return the synthesized answer

    Returns a unified trace dict matching Step 2 §2.4:
    {decision, reason, signals, intermediate, final_answer, trace_log}
    """

    def __init__(self, ablate=None):
        self.ablate = set(ablate or [])

        self.sufficiency = EvidenceSufficiencyAgent()
        self.groundedness = GroundednessAgent()
        self.contradiction = ContradictionAgent()
        self.clarification = ClarificationAgent()
        self.abstention = AbstentionAgent()
        self.critic = CriticAgent()
        self.recovery = RecoveryAgent()
        self.trust = TrustAgent()

        self._draft_answer = None
        self._last_strategy = None

    def retrieve(self, query, strategy="confidence", top_k=5):
        if strategy == "waterfall":
            answer, docs, trace = waterfall_orchestrate(query, top_k)
        elif strategy == "voting":
            answer, docs, trace = voting_orchestrate(query, top_k)
        else:
            answer, docs, trace = confidence_orchestrate(query, top_k)

        self._draft_answer = answer
        self._last_strategy = strategy
        return docs, trace

    def generate_answer(self, docs):
        if len(docs) == 0:
            return "NOT FOUND"

        # Use orchestrator's synthesized answer if available
        if self._draft_answer and len(self._draft_answer.strip()) > 5:
            return self._draft_answer

        # Fallback: truncate first doc (legacy placeholder)
        return docs[0].page_content[:250]

    _YEAR_RE = re.compile(r'\b(1\d{3}|20\d{2})\b')

    def _extract_year(self, query):
        """Simple year extractor for temporal coherence checks."""
        m = self._YEAR_RE.search(query)
        return int(m.group()) if m else None

    def _ablated(self, agent_name):
        return agent_name in self.ablate

    def _compute_signals(self, query, docs, answer, ablated, query_type=None):
        """Run all reliability agents and return signals dict."""
        trace_log = []

        suff = self.sufficiency.assess(query, docs)
        trace_log.append(f"Sufficiency: {suff['sufficient']} (score={suff['score']})")

        if not "groundedness" in ablated:
            grounded = self.groundedness.verify(answer, docs)
            trace_log.append(f"Groundedness: {grounded}")
        else:
            grounded = True
            trace_log.append("Groundedness: ABATED")

        if not "contradiction" in ablated:
            contradiction = self.contradiction.detect(docs)
            trace_log.append(f"Contradiction: {contradiction['contradiction']}")
        else:
            contradiction = {"contradiction": False}
            trace_log.append("Contradiction: ABATED")

        trust = self.trust.score(suff, grounded, contradiction)
        trace_log.append(f"Trust: {trust['score']} ({trust['label']})")

        if not "critic" in ablated:
            critique = self.critic.critique(
                answer, grounded, contradiction,
                query_type=query_type, query_year=self._extract_year(query),
            )
            trace_log.append(f"Critique: {len(critique)} issues")
        else:
            critique = []
            trace_log.append("Critique: ABATED")

        abstain = self.abstention.should_abstain(trust)

        signals = {
            "evidence_sufficiency": round(float(suff["score"]), 3),
            "grounding_score": 1.0 if grounded else 0.0,
            "has_contradictions": contradiction["contradiction"],
            "query_ambiguous": self.clarification.needs_clarification(query),
            "trust_score": trust["score"],
        }

        return {
            "suff": suff,
            "grounded": grounded,
            "contradiction": contradiction,
            "trust": trust,
            "critique": critique,
            "abstain": abstain,
            "signals": signals,
            "trace_log": trace_log,
        }

    def run(self,
            query,
            strategy="confidence",
            ablate=None,
            query_type=None,
            query_year=None):

        ablated = set(ablate or []) | self.ablate
        trace_log = []
        retry_count = 0
        recovery_action = None
        strategy_used = strategy

        # --- Clarification branch ---
        if self.clarification.needs_clarification(query):
            trace_log.append("Clarification triggered")
            return {
                "decision": "clarify",
                "reason": "Query is ambiguous (short or contains pronouns)",
                "signals": {
                    "evidence_sufficiency": 0.0,
                    "grounding_score": 0.0,
                    "has_contradictions": False,
                    "query_ambiguous": True,
                    "trust_score": 0.0,
                },
                "intermediate": {
                    "strategy_used": strategy,
                    "recovery_action": None,
                    "retry_count": 0,
                },
                "final_answer": None,
                "trace_log": trace_log,
            }

        # --- First retrieval pass ---
        docs, retrieval_trace = self.retrieve(query, strategy)
        trace_log.extend(retrieval_trace)
        answer = self.generate_answer(docs)

        result = self._compute_signals(query, docs, answer, ablated, query_type)
        trace_log.extend(result["trace_log"])

        # --- Recovery branch: try to RESCUE before abstaining ---
        # Recovery now runs WHEN the system would abstain (low reliability),
        # giving the pipeline a chance to fix the problem before giving up.
        if result["abstain"] and not "recovery" in ablated:
            recovery = self.recovery.recover(
                query, strategy, result["suff"], result["contradiction"],
                result["grounded"]
            )
            recovery_action = recovery["action"]
            trace_log.append(f"Low reliability -> recovery action: {recovery_action}")

            if recovery_action == "switch_strategy":
                new_strategy = recovery["new_strategy"]
                trace_log.append(f"RETRY: switching to {new_strategy}")
                docs, retrieval_trace = self.retrieve(query, new_strategy)
                trace_log.extend(retrieval_trace)
                answer = self.generate_answer(docs)
                result = self._compute_signals(query, docs, answer, ablated, query_type)
                trace_log.extend(result["trace_log"])
                retry_count = 1
                strategy_used = new_strategy

            elif recovery_action == "rewrite_query":
                rewritten = recovery["query"]
                trace_log.append(f"RETRY: rewritten query -> {rewritten}")
                docs, retrieval_trace = self.retrieve(rewritten, strategy)
                trace_log.extend(retrieval_trace)
                answer = self.generate_answer(docs)
                result = self._compute_signals(query, docs, answer, ablated, query_type)
                trace_log.extend(result["trace_log"])
                retry_count = 1

        # --- Abstention branch (runs after any recovery attempt) ---
        if result["abstain"]:
            if retry_count > 0:
                abstain_reason = "Recovery attempted but evidence still unreliable"
            else:
                abstain_reason = "Evidence insufficient, ungrounded, or contradictory"
            trace_log.append("System abstained")
            return {
                "decision": "abstain",
                "reason": abstain_reason,
                "signals": result["signals"],
                "intermediate": {
                    "strategy_used": strategy_used,
                    "recovery_action": recovery_action,
                    "retry_count": retry_count,
                },
                "final_answer": None,
                "trace_log": trace_log,
            }

        # --- Answer branch ---
        if retry_count > 0:
            answer_reason = "Recovery succeeded; all reliability checks passed"
        else:
            answer_reason = "All reliability checks passed"
        trace_log.append("Answer generated successfully")
        return {
            "decision": "answer",
            "reason": answer_reason,
            "signals": result["signals"],
            "intermediate": {
                "strategy_used": strategy_used,
                "recovery_action": recovery_action,
                "retry_count": retry_count,
            },
            "final_answer": answer,
            "trace_log": trace_log,
        }



## 7. Initialize System

In [19]:
rag_system = ReliableAdaptiveRAG()

## 8. Example Queries
This section demonstrates:

- clarification behavior
- abstention
- recovery
- contradiction handling
- successful answer generation

In [20]:


queries = [
    "Who received ERC grants at ETH?",
    "How does ETH support innovation?",
    "it",
    "What research areas are important at ETH Zurich?"
]

for q in queries:

    print("\n" + "=" * 80)
    print("QUERY:", q)

    result = rag_system.run(q)

    print("\nDECISION:", result["decision"])
    print("REASON:", result["reason"])
    print("ANSWER:", result["final_answer"] or "(none)")

    print("\nSIGNALS:")
    for k, v in result["signals"].items():
        print(f"  {k}: {v}")

    print("\nINTERMEDIATE:")
    for k, v in result["intermediate"].items():
        print(f"  {k}: {v}")

    print("\nTRACE:")
    for t in result["trace_log"]:
        print("-", t)


QUERY: Who received ERC grants at ETH?

DECISION: abstain
REASON: Recovery attempted but evidence still unreliable
ANSWER: (none)

SIGNALS:
  evidence_sufficiency: 0.6
  grounding_score: 1.0
  has_contradictions: True
  query_ambiguous: False
  trust_score: 0.26

INTERMEDIATE:
  strategy_used: voting
  recovery_action: switch_strategy
  retry_count: 1

TRACE:
- query
- query_type
- weights
- retry_weights
- gated_out
- retriever_counts
- retrieval_errors
- zero_result_retrievers
- retry_triggered
- critic_ok
- critic_feedback
- evidence_ids
- latency_s
- bilingual_variants
- Sufficiency: True (score=0.6)
- Groundedness: True
- Contradiction: True
- Trust: 0.26 (LOW)
- Critique: 1 issues
- Low reliability -> recovery action: switch_strategy
- RETRY: switching to voting
- query
- query_type
- weights
- retry_weights
- gated_out
- retriever_counts
- retrieval_errors
- zero_result_retrievers
- retry_triggered
- critic_ok
- critic_feedback
- evidence_ids
- latency_s
- bilingual_variants
- 

## 9. Benchmark Evaluation
This section evaluates:

- reliability scores
- latency
- system behavior
- abstention frequency

In [21]:
from pathlib import Path
results = []

# Use the real benchmark for project evaluation and failure analysis.
if "eval_qa_data" in globals() and eval_qa_data:
    _qa_data = eval_qa_data
elif "qa_data" in globals() and qa_data:
    _qa_data = qa_data
else:
    benchmark_path = Path("memory/benchmark_qa.csv")
    if not benchmark_path.exists():
        raise FileNotFoundError(
            "Could not find benchmark data. Expected memory/benchmark_qa.csv. "
            "Run from the repository root after the setup cell."
        )
    benchmark_source_df = pd.read_csv(benchmark_path)
    _qa_data = benchmark_source_df.to_dict("records")
    print(f"Loaded benchmark from {benchmark_path}: {len(_qa_data)} questions")

N = len(_qa_data)
print(f"Starting benchmark for {N} questions...")

for item in _qa_data:
    q = item.get("question") or item.get("query")
    if not q:
        continue

    start = time.time()
    result = rag_system.run(q)
    runtime = time.time() - start

    results.append({
        "query": q,
        "decision": result.get("decision"),
        "trust": result.get("signals", {}).get("trust_score"),
        "runtime": runtime,
        "strategy": result.get("intermediate", {}).get("strategy_used"),
        "retry_count": result.get("intermediate", {}).get("retry_count"),
    })

benchmark_df = pd.DataFrame(
    results,
    columns=["query", "decision", "trust", "runtime", "strategy", "retry_count"],
)
display(benchmark_df.head())


Loaded benchmark from memory/benchmark_qa.csv: 25 questions
Starting benchmark for 25 questions...


,query,decision,trust,runtime,strategy,retry_count
0,who was president of eth in 2003?,answer,0.728,0.286699,confidence,0
1,who were the rectors of eth between 2017 and 2...,answer,0.624,0.114415,confidence,0
2,who at eth received erc grants?,abstain,0.240,0.267873,voting,1
3,when did the insight get to mars?,abstain,0.243,0.500178,voting,1
4,what did prof. schubert say about flying?,abstain,0.106,0.478966,voting,1


## 9a. Export Audit CSV

Run the cell below to export every query's full reliability data to a **timestamped CSV file**.

This creates an audit trail so you can:
- Compare runs after changing thresholds or agents
- Share results with teammates without screenshots
- Roll back to a known-good configuration

Filename format: `YYYY-MM-DD_HH-MM-SS_output_step3.csv`


In [22]:
# --- CSV Export for Audit Trail ---
# Run this after the benchmark cell to save all reliability data to a timestamped CSV.

import csv, time, os
from datetime import datetime

audit_rows = []

# Re-run benchmark with FULL signal extraction + error handling
if "eval_qa_data" in globals() and eval_qa_data:
    _qa_data = eval_qa_data
elif "qa_data" in globals() and qa_data:
    _qa_data = qa_data
else:
    from pathlib import Path
    benchmark_path = Path("memory/benchmark_qa.csv")
    if not benchmark_path.exists():
        raise FileNotFoundError(
            "Could not find benchmark data. Expected memory/benchmark_qa.csv. "
            "Run from the repository root after the setup cell."
        )
    _qa_data = pd.read_csv(benchmark_path).to_dict("records")
    print(f"Loaded benchmark from {benchmark_path}: {len(_qa_data)} questions")

print(f"Exporting {len(_qa_data)} queries to CSV...")

run_timestamp = datetime.now().isoformat()

for i, item in enumerate(_qa_data):
    q = item["question"]
    start = time.time()

    try:
        result = rag_system.run(q)
        runtime = time.time() - start
    except Exception as e:
        print(f"Query {i} crashed: {q[:50]}... | Error: {e}")
        runtime = time.time() - start
        result = {
            "decision": "error",
            "reason": str(e),
            "signals": {},
            "intermediate": {},
            "final_answer": None,
            "trace_log": [f"ERROR: {e}"],
        }

    sig = result.get("signals", {})
    inter = result.get("intermediate", {})

    row = {
        "query": q,
        "decision": result.get("decision", ""),
        "reason": result.get("reason", ""),
        "final_answer": result.get("final_answer") or "",
        "trust_score": sig.get("trust_score", 0.0),
        "evidence_sufficiency": sig.get("evidence_sufficiency", 0.0),
        "grounding_score": sig.get("grounding_score", 0.0),
        "has_contradictions": sig.get("has_contradictions", False),
        "query_ambiguous": sig.get("query_ambiguous", False),
        "strategy_used": inter.get("strategy_used", ""),
        "recovery_action": inter.get("recovery_action", ""),
        "retry_count": inter.get("retry_count", 0),
        "runtime_sec": round(runtime, 4),
        "run_timestamp": run_timestamp,
        "trace_summary": " | ".join(result.get("trace_log", [])[:5]),
    }
    audit_rows.append(row)

# Build filename with timestamp
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename = f"{ts}_output_step3.csv"
filepath = f"/content/{filename}" if os.path.exists("/content") else filename

# Write CSV
if audit_rows:
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=audit_rows[0].keys())
        writer.writeheader()
        writer.writerows(audit_rows)
    print(f"Saved {len(audit_rows)} rows to: {filepath}")
    print("")
    print("Preview:")
    display(pd.DataFrame(audit_rows).head())
else:
    print("No rows to export.")


Loaded benchmark from memory/benchmark_qa.csv: 25 questions
Exporting 25 queries to CSV...
Saved 25 rows to: /content/2026-06-14_16-50-07_output_step3.csv

Preview:


,query,decision,reason,final_answer,trust_score,evidence_sufficiency,grounding_score,has_contradictions,query_ambiguous,strategy_used,recovery_action,retry_count,runtime_sec,run_timestamp,trace_summary
0,who was president of eth in 2003?,answer,All reliability checks passed,"in may 2023, pilloud was elected president of ...",0.728,0.714,1.0,False,False,confidence,None,0,0.2427,2026-06-14T16:50:02.910483,query | query_type | weights | retry_weights |...
1,who were the rectors of eth between 2017 and 2...,answer,All reliability checks passed,the [vice rector for continuing education](htt...,0.624,0.540,1.0,False,False,confidence,None,0,0.1063,2026-06-14T16:50:02.910483,query | query_type | weights | retry_weights |...
2,who at eth received erc grants?,abstain,Recovery attempted but evidence still unreliable,,0.240,0.567,1.0,True,False,voting,switch_strategy,1,0.2704,2026-06-14T16:50:02.910483,query | query_type | weights | retry_weights |...
3,when did the insight get to mars?,abstain,Recovery attempted but evidence still unreliable,,0.243,0.571,1.0,True,False,voting,switch_strategy,1,0.2398,2026-06-14T16:50:02.910483,query | query_type | weights | retry_weights |...
4,what did prof. schubert say about flying?,abstain,Recovery attempted but evidence still unreliable,,0.106,0.343,1.0,True,False,voting,switch_strategy,1,0.2336,2026-06-14T16:50:02.910483,query | query_type | weights | retry_weights |...


## 10. Aggregate Statistics


In [23]:
# Summarize benchmark results by decision.
# Robust against stale/empty benchmark_df objects and column-name whitespace.
if "benchmark_df" not in globals():
    print("benchmark_df does not exist. Run the benchmark cell above first.")
else:
    benchmark_df = benchmark_df.copy()
    benchmark_df.columns = [str(c).strip() for c in benchmark_df.columns]

    required_cols = {"decision", "trust", "runtime"}
    missing_cols = sorted(required_cols - set(benchmark_df.columns))

    if benchmark_df.empty:
        print("benchmark_df is empty. Re-run the benchmark cell above first.")
        print("Available columns:", list(benchmark_df.columns))
    elif missing_cols:
        print("benchmark_df is missing required columns:", missing_cols)
        print("Available columns:", list(benchmark_df.columns))
        display(benchmark_df.head())
    else:
        display(benchmark_df.groupby("decision", dropna=False)[["trust", "runtime"]].mean())


,trust,runtime
decision,,
abstain,0.158000,0.321153
answer,0.615462,0.182292
clarify,0.000000,0.000013


In [24]:
if benchmark_df.empty or "decision" not in benchmark_df.columns:
    print("benchmark_df has no decision column. Re-run the benchmark cell above first.")
else:
    display(benchmark_df["decision"].value_counts())


,count
decision,
answer,13
abstain,9
clarify,3


## 11. Failure Analysis

This section examines low-confidence cases.

In [25]:
if benchmark_df.empty or "trust" not in benchmark_df.columns:
    print("benchmark_df has no trust column. Re-run the benchmark cell above first.")
else:
    low_conf = benchmark_df[benchmark_df["trust"] < 0.4]
    display(low_conf)


,query,decision,trust,runtime,strategy,retry_count
2,who at eth received erc grants?,abstain,0.240,0.267873,voting,1
3,when did the insight get to mars?,abstain,0.243,0.500178,voting,1
4,what did prof. schubert say about flying?,abstain,0.106,0.478966,voting,1
5,what is e-sling?,clarify,0.000,0.000009,confidence,0
8,how do alpine plants respond to climate change?,abstain,0.320,0.280563,voting,1
9,how would you make ferzlizer without carbon em...,abstain,0.000,0.275556,voting,1
10,what research is eth famous for?,abstain,0.280,0.271781,voting,1
11,how much of eth’s electricity consumpzon is du...,clarify,0.000,0.000019,confidence,0
13,who is working on methods for targeted cancer ...,clarify,0.000,0.000011,confidence,0
17,what conneczons does eth have to hong kong?,abstain,0.125,0.277213,voting,1


## 12. Final Discussion

This notebook implemented a reliable and adaptive agentic RAG framework aligned with the official Step 3 requirements.

Implemented capabilities:

- evidence sufficiency estimation
- groundedness verification
- contradiction detection
- clarification handling
- abstention
- self-reflection
- adaptive recovery
- trust estimation

The system dynamically adapts its behavior when failures or uncertainty are detected.

Limitations:

- heuristic-based reliability estimation
- lightweight contradiction detection
- simple answer generation

Future improvements:

- stronger verifier models
- semantic contradiction detection
- reinforcement learning orchestration
- retrieval strategy optimization
- claim-level grounding verification